# 課程紀錄
此程式可以相對容易的從全人系統上把修課紀錄抓下來
需要安裝以下依賴
- requests
- pandas
- beautifulsoup4

In [15]:
import csv
import json

course_information = set()
with open('course_information.csv', mode='r', newline='') as file:
    reader = csv.DictReader(file)
    for row in reader:
        course_information.add(row["course_id"])

## 使用步驟
1. 登入政大全人系統
2. 拉到最底下點選資料格式化匯出
3. 勾選課業學習並下載
4. 把下載檔案放入tools資料夾中

In [ ]:
course_records = []

with open("exportStudentData.json", "r",) as f:
    data = json.load(f)

for dict in data:
    if "課業學習" in dict.keys():
        student_data = dict["課業學習"]

for grade_record in student_data["gradeRecordList"]:
    for course in grade_record["GradeRecords"]:
        if course["courseCode"] in course_information:
            grade = 0
            status = ""
            try:
                if float(course["score"]) >= 60:
                    status = "passed"
                else:
                    status = "failed"
                grade = course["score"]
            except:
                match course["score"]:
                    case "成績未到或無成績":
                        if course["credit"] == "0.0":
                            status = "passed"
                        else:
                            status = "unknown"
                    case "通過":
                        status = "passed"
                    case "停修":
                        status = "failed"
                grade = ""
                
            course_records.append([
                student_data["aboutMe"]["studentNumber"],
                course["courseCode"],
                f"{course["academicYear"]}-{course["semester"]}",
                grade,
                status
            ])

with open('course_record.csv', 'w', encoding='utf-8', newline="") as f:
    writer = csv.writer(f)
    writer.writerows(course_records)